In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración básica de visualización
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Crear carpeta para guardar datos procesados
ruta_datos_limpios = Path("data/processed")
ruta_datos_limpios.mkdir(parents=True, exist_ok=True)

# Cargar dataset original
ruta_consumos = "../data/raw/consumos_uptc.csv"
df_consumos = pd.read_csv(ruta_consumos)

# Convertir columna de fecha
df_consumos["timestamp"] = pd.to_datetime(df_consumos["timestamp"])

cantidad_registros = len(df_consumos)
filas, columnas = df_consumos.shape

print(f"Dataset cargado. Filas: {filas} | Columnas: {columnas}")
print(f"Registros totales: {cantidad_registros:,}")

# Guardar cantidad inicial de registros para control
registros_originales = cantidad_registros

Dataset cargado: (275387, 26)
Registros: 275,387


In [ ]:
# Corregir valores negativos (energía y CO2)
print("1. Corregir valores negativos")

cantidad_negativos_energia = (df_consumos["energia_total_kwh"] < 0).sum()
cantidad_negativos_co2 = (df_consumos["co2_kg"] < 0).sum()

print(f"Negativos en energía total: {cantidad_negativos_energia}")
print(f"Negativos en CO2: {cantidad_negativos_co2}")

df_consumos["energia_total_kwh"] = df_consumos["energia_total_kwh"].abs()
df_consumos["co2_kg"] = df_consumos["co2_kg"].abs()

cantidad_negativos_despues = (df_consumos["energia_total_kwh"] < 0).sum()
print(f"Negativos en energía después: {cantidad_negativos_despues}")


Negativos energia: 581
Negativos CO2: 543
Negativos después: 0


In [ ]:
# Recalcular total cuando no coincide con la suma de sectores

print("2. Coherencia entre total y suma de sectores")

columnas_sectores = [
    "energia_comedor_kwh",
    "energia_salones_kwh",
    "energia_laboratorios_kwh",
    "energia_auditorios_kwh",
    "energia_oficinas_kwh",
]

sectores_existentes = [col for col in columnas_sectores if col in df_consumos.columns]

# Nota: si algún sector viene nulo, para esta suma lo tratamos como 0
suma_sectores = df_consumos[sectores_existentes].fillna(0).sum(axis=1)

# Evitar división por cero: si la suma es 0, dejamos porcentaje en 0
diferencia_absoluta = (df_consumos["energia_total_kwh"] - suma_sectores).abs()
porcentaje_diferencia = np.where(
    suma_sectores != 0,
    (diferencia_absoluta / suma_sectores.abs()) * 100,
    0,
)

limite_recalculo = 15
casos_incoherentes = (porcentaje_diferencia > limite_recalculo).sum()
print(f"Registros con diferencia > {limite_recalculo}%: {casos_incoherentes}")

# Recalcular total en los casos incoherentes
mascara_recalculo = porcentaje_diferencia > limite_recalculo
df_consumos.loc[mascara_recalculo, "energia_total_kwh"] = suma_sectores[mascara_recalculo]

print(f"Total recalculado para {mascara_recalculo.sum()} registros")


Registros con diferencia > 15%: 3899
Total recalculado para 3899 registros


In [ ]:
# Filtrar outliers extremos en energia_total_kwh

print("3. Filtrar outliers extremos")

percentil_bajo = df_consumos["energia_total_kwh"].quantile(0.005)
percentil_alto = df_consumos["energia_total_kwh"].quantile(0.995)

print(f"Percentil 0.5%: {percentil_bajo:.2f} kWh")
print(f"Percentil 99.5%: {percentil_alto:.2f} kWh")

mascara_outliers = (df_consumos["energia_total_kwh"] < percentil_bajo) | (df_consumos["energia_total_kwh"] > percentil_alto)
cantidad_outliers = mascara_outliers.sum()
print(f"Outliers a eliminar: {cantidad_outliers}")

df_consumos = df_consumos[~mascara_outliers].copy()

print(f"Registros después de filtrar: {len(df_consumos):,}")
print(f"Eliminados: {registros_originales - len(df_consumos):,}")


Percentil 0.5%: 0.26 kWh
Percentil 99.5%: 24.51 kWh
Outliers a eliminar: 2670
Registros después de filtrar: 272,717
Eliminados: 2,670


In [ ]:
# Completar faltantes en sectores de energía con 0
print("4. Completar faltantes en sectores de energía")

sectores_energia = [
    "energia_comedor_kwh",
    "energia_salones_kwh",
    "energia_laboratorios_kwh",
    "energia_auditorios_kwh",
    "energia_oficinas_kwh",
]

sectores_existentes = [col for col in sectores_energia if col in df_consumos.columns]

for sector in sectores_existentes:
    faltantes_antes = df_consumos[sector].isnull().sum()
    if faltantes_antes > 0:
        print(f"{sector}: faltantes antes = {faltantes_antes}")
        df_consumos[sector] = df_consumos[sector].fillna(0)
        faltantes_despues = df_consumos[sector].isnull().sum()
        print(f"{sector}: faltantes después = {faltantes_despues}")

faltantes_totales = df_consumos[sectores_existentes].isnull().sum().sum()
print(f"Faltantes totales en sectores después: {faltantes_totales}")


energia_comedor_kwh: 8154 missing
  Llenados con 0
energia_auditorios_kwh: 13487 missing
  Llenados con 0

Missing después: 0


In [ ]:
# Completar faltantes en ocupación (ocupacion_pct)
print("Completar faltantes en ocupación")

if "ocupacion_pct" not in df_consumos.columns:
    print("No existe la columna 'ocupacion_pct'.")
else:
    faltantes_ocupacion = df_consumos["ocupacion_pct"].isnull().sum()
    print(f"Faltantes en ocupación antes: {faltantes_ocupacion}")

    if faltantes_ocupacion > 0:
        # 1) Mediana por sede y hora
        df_consumos["ocupacion_pct"] = df_consumos.groupby(["sede", "hora"])["ocupacion_pct"].transform(
            lambda serie: serie.fillna(serie.median())
        )

        # 2) Si quedan, mediana por sede
        df_consumos["ocupacion_pct"] = df_consumos.groupby("sede")["ocupacion_pct"].transform(
            lambda serie: serie.fillna(serie.median())
        )

        # 3) Si quedan casos raros, mediana global
        mediana_ocupacion = df_consumos["ocupacion_pct"].median()
        df_consumos["ocupacion_pct"] = df_consumos["ocupacion_pct"].fillna(mediana_ocupacion)

    print(f"Faltantes en ocupación después: {df_consumos['ocupacion_pct'].isnull().sum()}")


Missing en ocupacion: 32451
Missing después: 0


In [ ]:
# temperatura_exterior_c
print("Completar faltantes en temperatura")

if "temperatura_exterior_c" not in df_consumos.columns:
    print("No existe la columna 'temperatura_exterior_c'.")
else:
    faltantes_temperatura = df_consumos["temperatura_exterior_c"].isnull().sum()
    print(f"Faltantes en temperatura antes: {faltantes_temperatura}")

    if faltantes_temperatura > 0:
        # 1) Mediana por sede y mes (si existe mes)
        if "mes" in df_consumos.columns:
            df_consumos["temperatura_exterior_c"] = df_consumos.groupby(["sede", "mes"])["temperatura_exterior_c"].transform(
                lambda serie: serie.fillna(serie.median())
            )

        # 2) Si quedan, mediana por sede
        df_consumos["temperatura_exterior_c"] = df_consumos.groupby("sede")["temperatura_exterior_c"].transform(
            lambda serie: serie.fillna(serie.median())
        )

        # 3) Fallback final: mediana global
        mediana_temperatura = df_consumos["temperatura_exterior_c"].median()
        df_consumos["temperatura_exterior_c"] = df_consumos["temperatura_exterior_c"].fillna(mediana_temperatura)

    print(f"Faltantes en temperatura después: {df_consumos['temperatura_exterior_c'].isnull().sum()}")


Missing en temperatura: 16227
Missing después: 0


In [ ]:
print("Completar faltantes en agua")

if "agua_litros" not in df_consumos.columns:
    print("No existe la columna 'agua_litros'.")
else:
    faltantes_agua = df_consumos["agua_litros"].isnull().sum()
    print(f"Faltantes en agua antes: {faltantes_agua}")

    if faltantes_agua > 0:
        # Mediana por sede
        df_consumos["agua_litros"] = df_consumos.groupby("sede")["agua_litros"].transform(
            lambda serie: serie.fillna(serie.median())
        )

        # Fallback final: mediana global
        mediana_agua = df_consumos["agua_litros"].median()
        df_consumos["agua_litros"] = df_consumos["agua_litros"].fillna(mediana_agua)

    print(f"Faltantes en agua después: {df_consumos['agua_litros'].isnull().sum()}")


Missing en agua: 10860
Missing después: 0


In [ ]:
print("Recalcular CO2 desde energía")

factor_co2_por_kwh = 0.45

if "co2_kg" not in df_consumos.columns:
    print("No existe la columna 'co2_kg'. Se creará a partir de energía.")
    df_consumos["co2_kg"] = df_consumos["energia_total_kwh"] * factor_co2_por_kwh
else:
    df_consumos["co2_kg"] = df_consumos["energia_total_kwh"] * factor_co2_por_kwh

print(f"Faltantes en CO2 después: {df_consumos['co2_kg'].isnull().sum()}")


CO2 recalculado para todos los registros
Missing en CO2: 0


In [ ]:
print("5. Estandarizar periodo_academico")

if "periodo_academico" not in df_consumos.columns:
    print("No existe la columna 'periodo_academico'.")
else:
    print("Valores únicos antes:")
    print(df_consumos["periodo_academico"].value_counts(dropna=False))

    # Normalizar texto
    df_consumos["periodo_academico"] = df_consumos["periodo_academico"].astype("string").str.strip().str.lower()

    # Consolidar variantes conocidas
    mapeo_periodo = {
        "semestre_1": "semestre_1",
        "semestre1": "semestre_1",
        "semestre 1": "semestre_1",
        "semestre_2": "semestre_2",
        "semestre2": "semestre_2",
        "semestre 2": "semestre_2",
    }

    df_consumos["periodo_academico"] = df_consumos["periodo_academico"].replace(mapeo_periodo)

    print("Valores únicos después:")
    print(df_consumos["periodo_academico"].value_counts(dropna=False))


Valores únicos antes:
periodo_academico
semestre_1          114663
semestre_2           78416
vacaciones_mitad     50567
vacaciones_fin       26312
SEMESTRE_1             716
Semestre_1             703
semestre1              693
vacaciones             647
Name: count, dtype: int64

Valores únicos después:
periodo_academico
semestre_1          116775
semestre_2           78416
vacaciones_mitad     50567
vacaciones_fin       26312
vacaciones             647
Name: count, dtype: int64


In [ ]:
# Crear variables para analisis/modelado
print("6. Crear variables derivadas")

# Hora pico: franjas típicas de alto movimiento
df_consumos["hora_pico"] = df_consumos["hora"].isin([7, 8, 9, 10, 11, 14, 15, 16, 17]).astype(int)

# Vacaciones: si el texto del periodo menciona vacaciones
if "periodo_academico" in df_consumos.columns:
    df_consumos["es_vacaciones"] = df_consumos["periodo_academico"].fillna("").str.contains("vacaciones", regex=False).astype(int)
else:
    df_consumos["es_vacaciones"] = 0

# Nocturno: 0 a 6
df_consumos["es_nocturno"] = df_consumos["hora"].isin([0, 1, 2, 3, 4, 5, 6]).astype(int)

# Pandemia: usa columna año/año según exista
columna_anio = None
if "año" in df_consumos.columns:
    columna_anio = "año"
elif "anio" in df_consumos.columns:
    columna_anio = "anio"

if columna_anio is not None:
    df_consumos["anio_pandemia"] = df_consumos[columna_anio].isin([2020, 2021]).astype(int)
else:
    df_consumos["anio_pandemia"] = 0

print("Distribución de variables derivadas:")
print(f"Hora pico: {df_consumos['hora_pico'].sum()} registros")
print(f"Vacaciones: {df_consumos['es_vacaciones'].sum()} registros")
print(f"Nocturno: {df_consumos['es_nocturno'].sum()} registros")
print(f"Pandemia: {df_consumos['anio_pandemia'].sum()} registros")


Features derivadas creadas:
  hora_pico: 101580 registros en pico
  es_vacaciones: 77526 registros en vacaciones
  es_nocturno: 79809 registros nocturnos
  anio_pandemia: 69086 registros en pandemia


In [ ]:
print("Validación final")

filas, columnas = df_consumos.shape
print(f"Filas: {filas} | Columnas: {columnas}")
print(f"Registros: {len(df_consumos):,}")

faltantes_totales = df_consumos.isnull().sum().sum()
print(f"Faltantes totales: {faltantes_totales}")

if faltantes_totales > 0:
    print("Columnas con faltantes:")
    faltantes_por_columna = df_consumos.isnull().sum()
    print(faltantes_por_columna[faltantes_por_columna > 0])

negativos_energia = (df_consumos["energia_total_kwh"] < 0).sum()
print(f"Valores negativos en energía total: {negativos_energia}")

print("Resumen de retención:")
print(f"Registros originales: {registros_originales:,}")
print(f"Registros finales: {len(df_consumos):,}")
print(f"Retenidos: {len(df_consumos) / registros_originales * 100:.1f}%")
print(f"Eliminados: {registros_originales - len(df_consumos):,}")

Validación final:
Shape: (272717, 30)
Registros: 272,717

Total missing values: 0

Valores negativos en energia: 0

Registros originales: 275,387
Registros finales: 272,717
Retenidos: 99.0%
Eliminados: 2,670


In [ ]:
print("Guardar dataset limpio")

ruta_salida = ruta_datos_limpios / "consumos_uptc_clean.csv"
df_consumos.to_csv(ruta_salida, index=False)

tamano_mb = ruta_salida.stat().st_size / 1024 / 1024
print(f"Dataset limpio guardado en: {ruta_salida}")
print(f"Tamaño del archivo: {tamano_mb:.1f} MB")


Dataset limpio guardado en: ../data/processed/consumos_uptc_clean.csv
Tamaño del archivo: 47.7 MB


In [ ]:
import pandas as pd
print("Revisión rápida del dataset limpio")

ruta_archivo_limpio = "data/processed/consumos_uptc_clean.csv"
df_consumos_limpio = pd.read_csv(ruta_archivo_limpio)
df_consumos_limpio["timestamp"] = pd.to_datetime(df_consumos_limpio["timestamp"])

filas, columnas = df_consumos_limpio.shape
print(f"Dataset limpio. Filas: {filas} | Columnas: {columnas}")
print(f"Registros: {len(df_consumos_limpio):,}")

print("Energía total:")
print(df_consumos_limpio["energia_total_kwh"].describe())

if "ocupacion_pct" in df_consumos_limpio.columns:
    print("Ocupación:")
    print(df_consumos_limpio["ocupacion_pct"].describe())

if "temperatura_exterior_c" in df_consumos_limpio.columns:
    print("Temperatura:")
    print(df_consumos_limpio["temperatura_exterior_c"].describe())

print("Consumo por sede:")
print(
    df_consumos_limpio.groupby("sede")["energia_total_kwh"].agg(["mean", "median", "std", "min", "max"])
)

if "periodo_academico" in df_consumos_limpio.columns:
    print("Consumo por periodo académico:")
    print(
        df_consumos_limpio.groupby("periodo_academico")["energia_total_kwh"].agg(["mean", "median", "count"])
    )

print("Consumo por día de semana:")
mapa_dias = {
    0: "Lunes",
    1: "Martes",
    2: "Miércoles",
    3: "Jueves",
    4: "Viernes",
    5: "Sábado",
    6: "Domingo",
}
consumo_por_dia = df_consumos_limpio.groupby("dia_semana")["energia_total_kwh"].agg(["mean", "median"])
consumo_por_dia.index = consumo_por_dia.index.map(mapa_dias)
print(consumo_por_dia)

print("Distribución de variables derivadas:")
for columna in ["hora_pico", "es_vacaciones", "es_nocturno", "anio_pandemia"]:
    if columna in df_consumos_limpio.columns:
        cantidad = int(df_consumos_limpio[columna].sum())
        porcentaje = df_consumos_limpio[columna].mean() * 100
        print(f"{columna}: {cantidad:,} registros ({porcentaje:.1f}%)")


Estadísticas descriptivas (post-limpieza):
Dataset limpio: (272717, 30)
Registros: 272,717

Energia total:
count    272717.000000
mean          4.898250
std           4.891276
min           0.255000
25%           1.482000
50%           3.616000
75%           5.672000
max          24.506000
Name: energia_total_kwh, dtype: float64

Ocupación:
count    272717.000000
mean         22.441053
std          19.540652
min           1.300000
25%           7.600000
50%          16.100000
75%          30.500000
max          90.200000
Name: ocupacion_pct, dtype: float64

Temperatura:
count    272717.000000
mean         14.025867
std           4.811221
min           2.000000
25%          10.400000
50%          14.000000
75%          17.700000
max          26.000000
Name: temperatura_exterior_c, dtype: float64

Consumo por sede:
                  mean  median       std    min        max
sede                                                      
Chiquinquirá  2.454923  1.3840  2.366315  0.255  15.92683

In [ ]:
# Verificación rápida de columnas del dataset limpio
import pandas as pd

ruta_archivo_limpio = "data/processed/consumos_uptc_clean.csv"

# Cargar dataset limpio
df_consumos_limpio = pd.read_csv(ruta_archivo_limpio)
df_consumos_limpio["timestamp"] = pd.to_datetime(df_consumos_limpio["timestamp"])

filas, columnas = df_consumos_limpio.shape
print(f"Dataset limpio cargado. Filas: {filas} | Columnas: {columnas}")

print(f"\nColumnas disponibles ({columnas}):")
print(df_consumos_limpio.columns.tolist())


Dataset cargado: (272717, 30)

Columnas disponibles (30):
['reading_id', 'timestamp', 'sede', 'sede_id', 'energia_total_kwh', 'energia_comedor_kwh', 'energia_salones_kwh', 'energia_laboratorios_kwh', 'energia_auditorios_kwh', 'energia_oficinas_kwh', 'potencia_total_kw', 'agua_litros', 'temperatura_exterior_c', 'ocupacion_pct', 'hora', 'dia_semana', 'dia_nombre', 'mes', 'trimestre', 'año', 'periodo_academico', 'es_fin_semana', 'es_festivo', 'es_semana_parciales', 'es_semana_finales', 'co2_kg', 'hora_pico', 'es_vacaciones', 'es_nocturno', 'anio_pandemia']
